# Chapter 2: Query Translation

**Estimated time: 15 minutes**

Your chunks are perfect. Your embeddings are state of the science. Your vector search still returns the wrong document. Why? Because the user asked a vague question and your pipeline searched for that vague question literally.

In this notebook you will ask the same question three different ways. The direct vector search will give you one thin sentence. The translated version will give you a rich answer that covers three angles the user actually cared about. Same corpus, same embedding model, same LLM. What changes is the question that gets sent to the database.

Run every cell top to bottom. When you reach the "Try it yourself" section, change the parameters and rerun.

## What you will see in three minutes

You are going to ask this vague question as a user of SkillAgents AI would:

> Can I sign my team up for this?

That question is the kind of thing a product manager types into a support bot after skimming your landing page. It is short, casual, and maddeningly ambiguous. It could mean "how much does a team plan cost", "what extra features does a team get", "can I pay with a purchase order", or any combination of those.

You will run it through four pipelines, side by side.

- **Direct search** embeds the vague question as is and queries FAISS. The top score is mediocre (L2 distance around 1.29) and the LLM answer is one thin sentence with only the price.
- **Multi-query** asks the LLM to rewrite the vague question into three specific sub-questions, runs each one, merges the results, and dedupes. The top score drops to around 0.52, a massive improvement, and the LLM answer covers pricing, team features, and the escalation path to Enterprise.
- **HyDE** asks the LLM to draft a fake answer to the question first, embeds the fake answer, and searches with that. Different retrieval intuition, similar payoff.
- **Decomposition** handles the opposite problem, a complex multi-part question that needs to be split into independent pieces.

Same question. Same corpus. Same model. The difference is what you feed the embedding layer. That is query translation.

## Install the dependencies

Run the next cell once. In Colab it installs the Python packages fresh. Locally it assumes you already ran `pip install -r requirements.txt` in your virtual environment.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 60 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without
    # this, a stale repo from a prior session would keep running the old utils.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. Drop any cached utils.* so the
# next import reads the freshly cloned file, not a stale copy from an earlier
# session.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your OpenAI, LangSmith, and Cohere keys out of Colab secrets (you set these up in Chapter 1) and turned on LangSmith tracing for this notebook. Every LLM call and every retrieval call in this chapter gets logged automatically to a visual waterfall at [smith.langchain.com](https://smith.langchain.com). Open it in another tab after you run a few cells.

## Look at the corpus

The corpus is the same six documents from Chapter 1. Pricing, product guide, billing FAQ, error codes, refund policy, and one outdated blog post. Chapter 1 focused on the refund policy. This chapter focuses on the Team plan, which lives across three different documents.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()
for d in docs[:6]:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## Peek at the three target passages

Before we run anything, let us look at the exact three passages that, together, answer the vague question "Can I sign my team up for this?". Each lives in a different document.

In [ ]:
pricing = next(d for d in docs if d.metadata["source"] == "pricing.pdf")
product_guide = next(d for d in docs if d.metadata["source"] == "product_guide.md")
billing = next(d for d in docs if d.metadata["source"] == "billing_faq.md")

def show_section(doc, needle, length=500):
    start = doc.page_content.find(needle)
    if start == -1:
        print(f"[not found: {needle}]")
        return
    print(doc.page_content[start:start + length])
    print()

print("--- pricing.pdf: Team plan row")
show_section(pricing, "Team", length=320)

print("--- product_guide.md: How Team features work")
show_section(product_guide, "How Team features work", length=420)

print("--- billing_faq.md: Do you accept purchase orders?")
show_section(billing, "Do you accept purchase orders", length=340)

These are the three chunks a complete answer needs. Pricing tells you the Team plan is $4,999 per year for up to five seats. The product guide tells you the Team plan adds a private Slack channel and a quarterly manager debrief. The billing FAQ tells you that purchase orders are accepted on Team and Enterprise plans with Net-30 terms.

A user who types "Can I sign my team up for this?" needs all three pieces to make a decision. Watch how many of them each pipeline actually retrieves.

## Build the index once

Chapter 1 taught you to pick `chunk_size=500` with `chunk_overlap=50` for this corpus. We are going to reuse that exact index for every pipeline in this chapter, so the only thing changing across the experiments is the query, not the chunks.

In [ ]:
from utils.shared import build_index

# Same chunking strategy as Chapter 1. We are isolating the query variable now.
index = build_index(chunk_size=500, chunk_overlap=50)

question = "Can I sign my team up for this?"
print(f"Question: {question}")

## Pipeline 1: direct search

First, the naive approach. Take the vague question, embed it as is, run it against FAISS, return the top three chunks.

In [ ]:
from utils.shared import search, show_results

direct_results = search(index, question, k=3)
show_results(direct_results, question=question)

## What direct search missed

Look at the score column. The best match is somewhere around 1.29 under L2 distance. Chapter 1 established the rule: under 1.0 is a good match, above 1.0 is effectively off-topic. This result is off-topic.

What happened? The word "team" appears in five different documents and the embedding model cannot tell which document the user meant. The vague question pulls weakly on all of them. The top hit is probably the Team plan row in `pricing.pdf` but the distance score is poor and the two neighbors are incomplete.

Now watch the LLM try to answer from that.

In [ ]:
from utils.shared import generate_answer

answer_direct = generate_answer(direct_results, question)
print(answer_direct)

The answer is probably one thin sentence about the $4,999 price. The LLM worked with what it was given. The retrieval layer only gave it pricing, so pricing is all it said.

A user who reads that answer still does not know whether they get a private Slack channel, whether their manager gets a debrief, or whether they can pay by purchase order. The retrieval failure cascades into an answer that looks correct but is incomplete.

## Pipeline 2: multi-query, the hero technique

The fix is to stop sending the vague question to the database. Instead, ask an LLM to rewrite it first.

`generate_query_variants` takes the vague question plus a short product context and asks `gpt-4o-mini` to produce three specific sub-questions, each targeting a different angle. The product context is important. Without it, the LLM has no idea whether "this" means a software platform, a law firm, or a Pokemon card. With a one-sentence context, the rewrites are grounded.

In [ ]:
from utils.shared import generate_query_variants, show_queries

# A one-sentence description of the corpus. In production you would store this
# once per RAG index alongside the data. Pass it to every rewrite call.
product_context = (
    "SkillAgents AI is a live cohort-based course called AI Coding for "
    "Product Managers. The corpus contains pricing, billing, product guide, "
    "and refund policy documents."
)

variants = generate_query_variants(question, n=3, context=product_context)
show_queries(variants, title="LLM rewrites of the vague question")

Read the three rewrites. Each one names SkillAgents explicitly. Each one targets a distinct angle, probably something like cost, team features, and refund or enrollment. This is what production multi-query looks like. Specific, grounded, non-overlapping.

## Run all three sub-queries and merge

`multi_query_search` runs the original question AND each sub-question against the same FAISS index, dedupes results so the same chunk never shows up twice, and returns the merged list sorted by best distance score.

Including the original question is the standard production pattern. It guarantees multi-query never performs worse than direct search, even when the LLM rewriter drifts off-topic. At minimum you keep the direct hits. At best you add new angles the vague question alone could not reach.

In [ ]:
from utils.shared import multi_query_search

variants, multi_results = multi_query_search(
    index,
    question,
    n=3,
    k=3,
    context=product_context,
)

print(f"Retrieved {len(multi_results)} unique chunks across {len(variants)} sub-queries")
show_results(multi_results[:5], question=question)

## What multi-query got right

Two things changed in that table compared to the direct search.

First, the top distance score dropped from around 1.29 to around 0.52. That is not a small tweak. It is the difference between "the embedding model is guessing" and "the embedding model is confident". Lower distance means the retrieved chunks genuinely look like the sub-questions, not like a vague paraphrase.

Second, the top three sources are now different documents, not the same document three times. Each sub-question pulled in its own best match. Pricing is there. Product guide is there. The merge-and-dedupe step gave you a result set that covers multiple angles of the original question.

In [ ]:
answer_multi = generate_answer(multi_results[:5], question)
print(answer_multi)

## Read both answers side by side

Put the two answers next to each other. The direct answer is one sentence about the price. The multi-query answer probably covers the Team plan price, the private Slack channel, the quarterly manager debrief, and the Enterprise escalation path for companies with more than five seats.

Same corpus. Same LLM. Same prompt template inside `generate_answer`. The only thing that changed is which chunks reached the LLM. The multi-query version fed the LLM a complete set of facts. The LLM composed a complete answer because it had complete evidence.

This is the lesson that matters for a PM. Retrieval is upstream of generation, and upstream changes compound downstream. A better rewriter fixes retrieval, and fixed retrieval produces the answer the user actually needed.

## Pipeline 3: HyDE, hypothetical document embedding

HyDE flips the intuition. Instead of rewriting the question, you write a fake answer.

Why would that help? Because the user's question is short and vague. Real documents are long and specific. The embedding of a short vague question is far from the embedding of a real doc in vector space. But a hypothetical answer, even a wrong one, is structurally similar to a real doc. Its embedding lands in the same neighborhood.

In [ ]:
from utils.shared import generate_hyde_doc

hypo = generate_hyde_doc(question, context=product_context)
print(hypo)

Read the hypothetical passage above. It sounds like a help article. It uses the right vocabulary (seats, enrollment, team access). It is probably wrong on the specifics (the exact price, the exact number of seats) because the LLM is guessing. That does not matter. What matters is that it looks like a real doc, so its embedding lands near a real doc.

In [ ]:
from utils.shared import hyde_search

hypo, hyde_results = hyde_search(
    index,
    question,
    k=5,
    context=product_context,
)
show_results(hyde_results, question=question)

HyDE should produce scores in the 0.6 range, similar to multi-query. The top hits are probably pricing and product guide, which is a close match to where multi-query landed. Different technique, comparable payoff.

**When HyDE wins over multi-query.** HyDE shines when your documents follow a consistent style (help articles, legal paragraphs, API reference). The fake answer has a real style to aim at.

**When HyDE loses.** HyDE struggles when your documents are messy or highly structured (tables, lists, step-by-step guides). The fake answer cannot match a format that varies from page to page.

For most production systems, start with multi-query and add HyDE only if your docs are clean and your multi-query rewrites keep drifting away from real content.

## Pipeline 4: decomposition for complex multi-part questions

Multi-query and HyDE both handle vague questions. Decomposition handles the opposite problem, questions that are not vague at all but contain multiple independent parts.

In [ ]:
from utils.shared import decompose_question

complex_question = (
    "Compare the Student and Pro Annual plans and tell me which one "
    "is better for someone who wants to retake the course next year."
)

sub_questions = decompose_question(complex_question, n=3)
show_queries(sub_questions, title="Decomposed sub-questions")

The three sub-questions can each be answered on their own. The first one asks about Student, the second about Pro Annual, and the third about retake policy specifically. You retrieve independently for each, then ask the LLM to synthesize.

In practice, the decomposition pattern is most useful when a question contains the word "compare", "versus", "difference", or "both". Those words are signals that a single vector search cannot cover the full question.

In [ ]:
from utils.shared import search

# Retrieve for each sub-question and combine into one context.
combined = []
for sq in sub_questions:
    combined.extend(search(index, sq, k=2))

# Dedupe by source + first 120 chars, keep best distance per unique chunk.
seen = {}
for doc, score in combined:
    key = (doc.metadata.get("source"), doc.page_content[:120])
    existing = seen.get(key)
    if existing is None or score < existing[1]:
        seen[key] = (doc, float(score))

decomposed_results = sorted(seen.values(), key=lambda pair: pair[1])
show_results(decomposed_results[:5], question=complex_question)

In [ ]:
decomposed_answer = generate_answer(decomposed_results[:5], complex_question)
print(decomposed_answer)

The answer should cover Student, Pro Annual, and make a direct recommendation for a retaker. That is a complete answer to the original multi-part question. A single vector search on the full question would probably mix everything into one average and miss the retake angle.

## Pipeline 5: step-back prompting for narrow questions

Step-back goes the other direction. When a user asks a very narrow question, sometimes the best way to answer it is to retrieve broader background context first and reason from there.

In [ ]:
from utils.shared import stepback_question

narrow_question = "Can I pause my Pro Annual subscription for two months if I travel?"
broader = stepback_question(narrow_question)

print(f"Narrow:  {narrow_question}")
print(f"Broader: {broader}")

In [ ]:
narrow_results = search(index, narrow_question, k=3)
broader_results = search(index, broader, k=3)

print("NARROW search:")
for d, s in narrow_results:
    print(f"  {s:.3f}  {d.metadata['source']}")
print()
print("BROADER search:")
for d, s in broader_results:
    print(f"  {s:.3f}  {d.metadata['source']}")

The broader search probably pulls in `billing_faq.md` with the general pause policy and `refund_policy.pdf` with the deferral clause. Those two together give the LLM enough context to answer the narrow travel question correctly, even though neither doc mentions "two months" or "travel" specifically.

Step-back is the trick you reach for when the user asks for a specific detail that is not spelled out in your docs, but the general policy is. The broader query surfaces the policy, and the LLM reasons from the policy to the specific case.

## Which technique should you use when?

You have four knobs now. Here is how to pick.

| Technique | Use when | Skip when |
|---|---|---|
| **Multi-query** | The user asked a vague question that could mean several things. This is the default and fits most production RAG systems. | Your docs are so specific that any rewrite produces the same result. |
| **HyDE** | Your docs have a consistent house style and the user's question is short. The hypothetical answer will match real doc structure. | Your docs are messy, mixed formats, or highly structured (tables, code, lists). |
| **Decomposition** | The question contains multiple parts ("compare X and Y", "what is A and how does it relate to B"). | The question is a single clean ask. |
| **Step-back** | The user asked about a specific detail that is probably not in your docs, but a general policy is. | The user asked a question that should be answered literally from a specific chunk. |

In practice, most production RAG systems start with direct search, measure where it fails, add multi-query when they see vague queries, add decomposition when they see complex queries, and reach for HyDE or step-back only when the first two are not enough. Start simple. Measure what fails. Add complexity only where it helps.

## Open your LangSmith trace

Open [smith.langchain.com](https://smith.langchain.com), click into the project called `rag-for-pms`, and open the most recent trace. You will see every sub-query, every retrieval call, every similarity score, and every LLM call in one visual waterfall. Five minutes of clicking around the trace view will teach you more about what query translation is actually doing than an hour of reading this notebook.

## What you can do at work on Monday

Query translation is the single biggest fix you can make to a production RAG system that is already chunking well. Here are the three questions to ask on your next RAG review.

1. When a user asks a vague question, does the pipeline rewrite it before retrieval, or does it search the raw question? If the answer is "we search the raw question", that is your biggest miss.
2. How does the rewriter know what product the user is asking about? Is the product context injected on every call, or is the LLM guessing?
3. What happens when a user asks a compare-and-contrast question? Does retrieval happen once on the full question, or once per sub-question?

A team that cannot answer these questions cleanly is probably shipping retrieval results that look correct but are missing half the evidence. Every question is an invitation to open a LangSmith trace and look at the actual sub-queries and the actual retrieved chunks.

## Try it yourself

Pick any of these. Change the value in the relevant cell, rerun, watch what happens.

1. Change `n=3` to `n=5` in the `multi_query_search` call. Do the extra two sub-questions add meaningful coverage, or do they overlap with the first three? Open the printed variants and read them.
2. Change the `question` variable to `"How do I pay for this?"` and rerun the direct-search and multi-query cells. Watch which documents rise to the top under multi-query.
3. Delete the `context=product_context` argument from the `generate_query_variants` call and rerun. The rewrites will drift into generic software rollout questions. Compare the scores. This is the exact failure mode you are trying to avoid in production.

## Homework

Two exercises you can do on your own. Each takes about fifteen minutes.

1. **Write your own product context.** Pick a real product from your own company. Draft a one-sentence product context for its corpus, then write three vague user questions the way a real user would type them. Paste each one into `generate_query_variants` with your own context and read the rewrites. Are they specific enough? If not, rewrite the context and try again. This is the single most important prompt in a multi-query pipeline.

2. **Build your own decomposition test.** Take a complex multi-part question you have actually seen a user ask about your product ("compare A and B and tell me which is better for X"). Run it through `decompose_question` and look at the three sub-questions. Do they cover every part of the original? If one part is missing, what would you add to the decomposition prompt to catch it?

## What is next

In Chapter 3 your rewriter is perfect and your retrieval is clean, but your RAG system is still shipping the wrong answer to some users. The reason this time is not about how the question is phrased. It is about where the answer lives. Some questions belong in a vector database. Others belong in a SQL table. And one class of questions belongs in a graph.

Chapter 3 is about routing. Sending each question to the data source that actually knows the answer. See you there.